## Import packages

In [1]:
import os
import sys
import json
import argparse
import numpy as np
import math
from einops import rearrange
import time
import random
import string
import h5py
from tqdm import tqdm
import webdataset as wds
import gc

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import transforms

# tf32 data type is faster than standard float32
torch.backends.cuda.matmul.allow_tf32 = True

## Configuration

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
data_type = torch.float16 # change depending on your mixed_precision
num_devices = torch.cuda.device_count()
batch_size = 32
num_epochs=12

print(f"data_type={data_type}, num_devices={num_devices}, batch_size={batch_size}, num_epochs={num_epochs}")

data_path = "/teamspace/studios/this_studio/nsd"
subj = 1
subj_list = [subj]
num_sessions = 2
num_test = 1985
num_voxels_list = []

num_samples_per_epoch = (750 * num_sessions) // num_devices 
num_iterations_per_epoch = num_samples_per_epoch // (batch_size * len(subj_list))

def my_split_by_node(urls): 
    return urls

print(f"data_path={data_path}, subj={subj}, subj_list={subj_list}, num_sessions={num_sessions}, num_test={num_test}, num_voxels_list={num_voxels_list}")

data_type=torch.float16, num_devices=1, batch_size=32, num_epochs=12
data_path=/teamspace/studios/this_studio/nsd, subj=1, subj_list=[1], num_sessions=2, num_test=1985, num_voxels_list=[]


## Creating wds dataloader

In [3]:
train_data = {}
train_dl = {}
num_voxels = {}
voxels = {}

for s in subj_list:

    train_url = f"{data_path}/wds/subj0{s}/train/" + "{0.." + f"{num_sessions-1}" + "}.tar"

    print(train_url)
    
    train_data[f'subj0{s}'] = wds.WebDataset(train_url,resampled=True,nodesplitter=my_split_by_node)\
                        .shuffle(750, initial=1500, rng=random.Random(42))\
                        .decode("torch")\
                        .rename(behav="behav.npy", past_behav="past_behav.npy", future_behav="future_behav.npy", olds_behav="olds_behav.npy")\
                        .to_tuple(*["behav", "past_behav", "future_behav", "olds_behav"])

    train_dl[f'subj0{s}'] = torch.utils.data.DataLoader(train_data[f'subj0{s}'], batch_size=batch_size, shuffle=False, drop_last=True, pin_memory=True)

    f = h5py.File(f'{data_path}/betas_all_subj0{s}_fp32_renorm.hdf5', 'r')
    betas = f['betas'][:]
    betas = torch.Tensor(betas).to("cpu").to(data_type)
    num_voxels_list.append(betas[0].shape[-1])
    num_voxels[f'subj0{s}'] = betas[0].shape[-1]
    voxels[f'subj0{s}'] = betas

    print(f"num_voxels for subj0{s}: {num_voxels[f'subj0{s}']}")

print("Loaded all subj train dls and betas!\n")

test_url = f"{data_path}/wds/subj0{subj}/test/" + "0.tar"

test_data = wds.WebDataset(test_url,resampled=False,nodesplitter=my_split_by_node)\
                    .shuffle(750, initial=1500, rng=random.Random(42))\
                    .decode("torch")\
                    .rename(behav="behav.npy", past_behav="past_behav.npy", future_behav="future_behav.npy", olds_behav="olds_behav.npy")\
                    .to_tuple(*["behav", "past_behav", "future_behav", "olds_behav"])
test_dl = torch.utils.data.DataLoader(test_data, batch_size=num_test, shuffle=False, drop_last=True, pin_memory=True)

print(f"Loaded test dl for subj{subj}!\n")

/teamspace/studios/this_studio/nsd/wds/subj01/train/{0..1}.tar
num_voxels for subj01: 15724
Loaded all subj train dls and betas!

Loaded test dl for subj1!



In [4]:
f = h5py.File(f'{data_path}/coco_images_224_float16.hdf5', 'r')
images = f['images'][:batch_size] # if you go OOM you can remove the [:] so it isnt preloaded to cpu! (will require a few edits elsewhere though)
images = torch.Tensor(images).to(device).to(data_type)
images.shape

torch.Size([32, 3, 224, 224])

## Load models

### CLIP image embeddings model

In [5]:
import clip
from PIL import Image

clip_embedder, preprocess = clip.load("ViT-B/32", device=device)

In [6]:
# Function to preprocess and embed images using CLIP
# def get_clip_embeddings(images, clip_model):
#     embeddings = clip_model.encode_image(images)
#     return embeddings
# clip_embeddings = get_clip_embeddings(images, clip_embedder)

### ImageToBrain

In [7]:
# Input: Clip latents of image
# Output: Brain representation

"""
input_sizes will be (batch_size, 768 or whatever's the flattened representation of the clip embedding) and 
out_features will be (batch_size, num_voxels)
"""

class ImageToBrain(torch.nn.Module):

    def __init__(self, input_sizes, out_features):
        super(ImageToBrain, self).__init__()
        self.out_features = out_features
        self.linears = torch.nn.Linear(input_sizes, out_features)
    def forward(self, x):
        out = self.linears(x)
        return out

In [8]:
input_dim = clip_embedder.visual.input_resolution  # Size of the CLIP embedding
output_dim = num_voxels[f'subj0{subj}']  # Number of voxels as target output
model = ImageToBrain(input_dim, output_dim).to(device)

## Main

### Preprocess

In [9]:
train_dls = [train_dl[f'subj0{s}'] for s in subj_list]

def preprocess():
    voxel_iters = {} # empty dict because diff subjects have differing # of voxels
    image_iters = torch.zeros(num_iterations_per_epoch, batch_size*len(subj_list), 3, 224, 224).float()
    
    for s, train_dl in enumerate(train_dls):
        with torch.cuda.amp.autocast(dtype=data_type):
            for iter, (behav0, past_behav0, future_behav0, old_behav0) in enumerate(train_dl):
                image0 = images[behav0[:,0,0].cpu().long()].float()
                image_iters[iter,s*batch_size:s*batch_size+batch_size] = image0
                
                voxel0 = voxels[f'subj0{subj_list[s]}'][behav0[:,0,5].cpu().long()]
                voxel0 = torch.Tensor(voxel0)

                voxel_iters[f"subj0{subj_list[s]}_iter{iter}"] = voxel0

                if iter >= num_iterations_per_epoch-1:
                    break
    
    return voxel_iters, image_iters

voxel_iters, image_iters = preprocess()

../aten/src/ATen/native/cuda/IndexKernel.cu:92: operator(): block: [31,0,0], thread: [64,0,0] Assertion `index >= -sizes[i] && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:92: operator(): block: [31,0,0], thread: [65,0,0] Assertion `index >= -sizes[i] && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:92: operator(): block: [31,0,0], thread: [66,0,0] Assertion `index >= -sizes[i] && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:92: operator(): block: [31,0,0], thread: [67,0,0] Assertion `index >= -sizes[i] && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:92: operator(): block: [31,0,0], thread: [68,0,0] Assertion `index >= -sizes[i] && index < sizes[i] && "index out of bounds"` failed.
../aten/src/ATen/native/cuda/IndexKernel.cu:92: operator(): block: [31,0,0], thread: [69,0,0] Assertion `index

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Train Loop

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()  # Mean Squared Error Loss

def train():
    voxel_iters, image_iters = preprocess()
    
    for epoch in range(epochs):
        for train_i in range(num_iterations_per_epoch):
            with torch.cuda.amp.autocast(dtype=data_type):
                optimizer.zero_grad()
                loss=0.

                voxel_list = [voxel_iters[f"subj0{s}_iter{train_i}"].detach().to(device) for s in subj_list]
                image = image_iters[train_i].detach()
                image = image.to(device)

                clip_latent = clip_img_embedder(image)

                # todo: model takes in clip latents and outputs voxel representation
                voxel_ridge_list = [model.ridge(voxel_list[si],si) for si,s in enumerate(subj_list)]
                voxel_ridge = torch.cat(voxel_ridge_list, dim=0)

                # Compute the loss between the model's output and CLIP latent
                loss = criterion(clip_latent, voxel_ridge)

                loss.backward()
                optimizer.step()

### Eval Loop